In [ ]:
"""
Random Forest grid search on Topological Representations - Example: Betti Curves.

Output:
    - CSV with CV performance for all configurations
"""

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, make_scorer, fbeta_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler


# Configurations

BASE = "RIPS"

DIMENSIONS = ["H0", "H1", "H0H1"]
FOLDER_TEMPLATE = "BettiCurves/{}"

CV_SPLITS = 6
RANDOM_STATE = 0

USE_SCALER = True

GRID = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10],
}

OVERSAMPLING = [
    ("no_ros", False),
    ("ros", True),
]

OUT_CSV = "rf_grid.csv"


# LOAD DATA
def load_features(folder: Path):
    files = sorted([f for f in folder.iterdir() if f.is_file() and not f.name.startswith(".")])
    data = []
    for f in files:
        try:
            v = np.loadtxt(f).flatten()
            data.append(v)
        except:
            continue
    return data


def build_pipeline(use_ros, params):

    steps = []

    if USE_SCALER:
        steps.append(("scaler", StandardScaler()))

    if use_ros:
        steps.append(("ros", RandomOverSampler(random_state=RANDOM_STATE)))

    steps.append(("rf",
        RandomForestClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
            **params
        )
    ))

    return ImbPipeline(steps)


def iter_grid(grid):

    keys = list(grid.keys())
    values = list(grid.values())

    for comb in np.array(np.meshgrid(*values, indexing="ij")).reshape(len(keys), -1).T:
        params = {k: v for k, v in zip(keys, comb)}

        # fix None
        if "max_depth" in params and isinstance(params["max_depth"], float):
            if np.isnan(params["max_depth"]):
                params["max_depth"] = None

        yield params


f2 = make_scorer(fbeta_score, beta=2, zero_division=0)

scoring = {
    "roc_auc": "roc_auc",
    "accuracy": "accuracy",
    "f2": f2
}

cv = StratifiedKFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


# Main

rows = []

for dim in DIMENSIONS:

    dirR = Path(BASE) / FOLDER_TEMPLATE.format(dim) / "Relapse"
    dirNR = Path(BASE) / FOLDER_TEMPLATE.format(dim) / "NonRelapse"

    X_R = load_features(dirR)
    X_NR = load_features(dirNR)

    X = np.vstack(X_NR + X_R)
    y = np.array([0]*len(X_NR) + [1]*len(X_R))

    for name, use_ros in OVERSAMPLING:

        for params in iter_grid(GRID):

            clf = build_pipeline(use_ros, params)

            scores = cross_validate(
                clf, X, y,
                cv=cv,
                scoring=scoring,
                n_jobs=-1,
                return_train_score=False
            )

            preds = cross_val_predict(clf, X, y, cv=cv, n_jobs=-1)
            cm = confusion_matrix(y, preds)

            rows.append({
                "model": "RF",
                "dimension": dim,
                "oversampling": name,
                "auc": np.mean(scores["test_roc_auc"]),
                "accuracy": np.mean(scores["test_accuracy"]),
                "f2": np.mean(scores["test_f2"]),
                "cm": f"({cm[0,0]} {cm[0,1]}; {cm[1,0]} {cm[1,1]})",
                **{f"rf__{k}": v for k, v in params.items()}
            })

    print(f"Done: {dim}")


df = pd.DataFrame(rows).sort_values(
    ["dimension", "oversampling", "f2"],
    ascending=[True, True, False]
).reset_index(drop=True)

df.to_csv(OUT_CSV, index=False)

print("Saved:", OUT_CSV)


# Best config pero block
best = df.groupby(["dimension", "oversampling"], as_index=False).head(1)

print("\nBest configs:")
print(best[["dimension", "oversampling", "auc", "accuracy", "f2"]])